# Elite papers: title-and-abstract BGE-M3 embeddings

Run **Embedding Elite Paper Abstracts → Clustering Elite Papers Abstracts → Topic analysis Elite Papers Abstracts** with **Python (batill)**.

The canonical selection contains 292 PDF works. The approved inputs now cover **282 works**: 278 reviewed PDF abstracts and four external full abstracts. Ten works remain excluded without usable full abstracts. Every paper keeps its original Pxxx ID and PDF hash.

The encoder matches the 3,819-record abstract corpus: BGE-M3 at the same pinned revision, model-defined CLS pooling, 1,024 float32 dimensions, L2 normalization, no prompt, truncation or chunking. The complete input is title + blank line + cleaned abstract. Existing full-paper and large-abstract runs are preserved.

PDF text retains exact page/block/character evidence. The four approved full-abstract supplements are cached in `data/selected_abstracts/supplements/` and selected by `configs/selected_abstracts_supplements.json`. Their URLs, source records, checksums and content types appear in the audit. Short publisher descriptions are excluded: P066, P070, P180, P299 and P318. No introductions or generated summaries fill missing abstracts.

P263 uses its published 2009 article abstract from the Scopus export, corroborated by the University of Chicago repository. Its Scopus DOI is blank; identity was reviewed by title, authors, journal and publication details rather than pretending to have an exact DOI match.

Previous 287-paper results are archived. The inputs now exclude the five short descriptions; execute the embedding, clustering and topic notebooks in order to produce updated results.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, HTML
from html import escape

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/batill').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the batill project.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from batill.selected_abstracts import (
    prepare_selected_abstracts, save_preparation, load_selected_documents,
    reference_settings, embed_selected_abstracts, activate_selected_run, load_selected_embeddings,
)
from batill.storage import read_json, write_json
OVERRIDES_FILE = ROOT / 'configs/selected_abstracts_review.json'
reviewed = read_json(OVERRIDES_FILE) if OVERRIDES_FILE.exists() else {}

## 1. Reproduce the selected corpus and audit abstract coverage

The same PDF-hash exclusions and reviewed duplicate decisions are applied as in the PDF notebook. Original Pxxx IDs and PDF hashes are preserved. Titles use the extracted title with whitespace normalization and trailing footnote markers removed. Reviewed title spans replace filenames, journal headers and incomplete titles. Where PDF title extraction is damaged or absent, the already reviewed version title is used and recorded in `title_source`; this supplies no abstract. Text cleaning uses the same functions as the large abstract preparation.

**Ready** means a strong same-DOI PDF text match, reviewed exact PDF spans, or an explicitly approved external full abstract/short publisher description. An Abstract heading alone does not establish complete and clean boundaries. **Review** means a possible abstract needs confirmation. **Missing** means no usable abstract is established (including a reviewed absence); inspect the reason. Neither pending category is silently embedded.

In [2]:
ready, audit, preparation = prepare_selected_abstracts(ROOT, overrides=reviewed)
print(f"Selected works: {len(audit)} | Ready abstracts: {len(ready)} | Pending: {len(audit)-len(ready)}")
display(audit.groupby(['status', 'extraction_method']).size().rename('Papers').to_frame())
missing = audit.loc[audit.status.eq('missing')]
needs_review = audit.loc[audit.status.eq('review')]
print(f"Missing/unavailable abstracts: {len(missing)}")
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120):
    display(missing[['paper_id', 'title', 'source_filename', 'extraction_flags', 'review_reason']])
    print(f"Uncertain extraction requiring review: {len(needs_review)}")
    display(needs_review[['paper_id', 'title', 'source_filename', 'extraction_flags', 'review_reason']])
print('Approved external full abstracts:')
supplements = audit.loc[audit.abstract_source_kind.ne('pdf_abstract')]
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120):
    display(supplements[['paper_id', 'title', 'abstract_source_kind', 'abstract_word_count', 'abstract_source_url']])


Selected works: 292 | Ready abstracts: 282 | Pending: 10


Papers
status  extraction_method                                    
missing reviewed_no_abstract                               10
ready   external_full_abstract                              4
        pdf_spans_validated_against_same_doi_abstract     190
        reviewed_pdf_spans                                 88

Missing/unavailable abstracts: 10


,paper_id,title,source_filename,extraction_flags,review_reason
64,P066,Private Equity Investment in Behavioral Health Treatment Centers,Brown et al. - 2020 - Private Equity Investment in Behavioral Health Treatment Centers.pdf,no_abstract,Viewpoint: title is followed by the article narrative and interview-based discussion. No separate abstract is present.
68,P070,Expansion of Private Equity Involvement in Women's Health Care,Bruch et al. - 2020 - Expansion of Private Equity Involvement in Women's Health Care.pdf,no_abstract,Research letter: the PDF begins with the end of another article. The selected article begins at Expansion of Private...
150,P159,"Eileen Appelbaum and Rosemary Batt, Private Equity at Work: When Wall Street Manages Main Street, New York, Russel S...","Froud - 2015 - Review Symposium Eileen Appelbaum and Rosemary Batt, Private Equity at Work When Wall Street Manages ...",no_abstract,Review Symposium/book review: keywords and reviewer details are followed by a narrative book review. No separate abs...
158,P168,Physician Practice Management and Private Equity: Market Forces Drive Change,Gilreath et al. - 2019 - Physician Practice Management and Private Equity Market Forces Drive Change.pdf,no_abstract,Practice Management article: introductory narrative is followed by The Need for Scale and Sophistication in Practice...
159,P169,Gastroenterology Physician Practice Management and Private Equity: Thriving in Uncertain Times,Gilreath et al. - 2021 - Gastroenterology Physician Practice Management and Private Equity Thriving in Uncertain Tim...,no_abstract,Practice Management follow-up article: introductory narrative is followed by Gastroenterology Practice Consolidation...
167,P180,Potential Implications of Private Equity Investments in Health Care Delivery,Gondi und Song - 2019 - Potential Implications of Private Equity Investments in Health Care Delivery.pdf,no_abstract,Viewpoint: introductory narrative proceeds into What Is Private Equity? and the subsequent discussion. No separate a...
189,P204,Erratum The financial performance of reverse leveraged buyouts [Journal of Financial Economics 42 (1996) 293-332] 1,Holthausen und Larcker - 1998 - The financial performance of reverse leveraged buyouts [Journal of Financial Economi...,no_abstract,"The cached PDF is an erratum, containing a correction notice and a quoted passage to eliminate, not a separate abstr..."
272,P297,"Commentary on ""The Impact of Vertical Integration on Physician Behavior and Healthcare Delivery: Evidence from Gastr...",Shah - 2023 - Commentary on The Impact of Vertical Integration on Physician Behavior and Healthcare Delivery Evi.pdf,no_abstract,"Commentary: title, author details and Keywords are followed directly by commentary text. No separate abstract is pre..."
274,P299,Geographic Variation in Private Equity Penetration Across Select Office-Based Physician Specialties in the US,Singh et al. - 2022 - Geographic Variation in Private Equity Penetration Across Select Office-Based Physician Specia...,no_abstract,"Research Letter: title and authors are followed directly by Introduction, Methods and Results. No separate abstract ..."
291,P318,"Private Equity Acquisitions of Physician Medical Groups Across Specialties, 2013-2016","Zhu et al. - 2020 - Private Equity Acquisitions of Physician Medical Groups Across Specialties, 2013-2016.pdf",no_abstract,"Research Letter: title is followed by background, Methods and Results, with no separate abstract. Title recovered fr..."


Uncertain extraction requiring review: 0


,paper_id,title,source_filename,extraction_flags,review_reason


Approved external full abstracts:


,paper_id,title,abstract_source_kind,abstract_word_count,abstract_source_url
46,P047,Do Private Equity Owned Firms Have Better Management Practices?,external_full_abstract,100,https://www.aeaweb.org/articles?id=10.1257/aer.p20151000
212,P229,Leveraged Buyouts and Private Equity,external_full_abstract,109,https://www.aeaweb.org/articles?id=10.1257/jep.23.1.121
243,P263,Does Private Equity Create Wealth? The Effects of Private Equity and Derivatives on Corporate Governance,external_full_abstract,87,https://chicagounbound.uchicago.edu/uclrev/vol76/iss1/8/
260,P284,Beware of Venturing into Private Equity,external_full_abstract,177,https://www.aeaweb.org/articles?id=10.1257/jep.23.1.147


## 2. Inspect the exact PDF text and extraction evidence

Choose a Pxxx ID. The first five extracted pages are shown with block IDs, block types and full text. Compare the candidate with its surrounding text. Repository covers, author affiliations, acknowledgements, keywords and introductions must not be treated as abstract content. The recorded offsets refer to the cached extracted block, not cleaned-text offsets.

For an external supplement, the embedded text comes from its cached source record, not from PDF spans. Its source is displayed below; PDF blocks remain available for work-identity inspection.

In [3]:
INSPECT_PAPER_ID = (audit.loc[audit.status.ne('ready'), 'paper_id'].iloc[0]
                    if audit.status.ne('ready').any() else audit.paper_id.iloc[0])
row = audit.set_index('paper_id').loc[INSPECT_PAPER_ID]
document = read_json(ROOT / row.document_path)
print(INSPECT_PAPER_ID, row.title)
print('Status:', row.status, '| Method:', row.extraction_method)
print('Document SHA-256:', row.document_sha256)
print('Candidate:', row.abstract)
print('Selected spans:', row.abstract_spans)
print('Abstract source:', row.abstract_source_kind, row.abstract_source_url)
print('Cached source:', row.abstract_source_path)
for block in document['blocks']:
    if block.get('page') is not None and block['page'] <= 5 and block['text'].strip():
        heading = escape(f"Page {block['page']} | {block['type']} | {block['id']}")
        display(HTML(f'<details><summary>{heading}</summary><pre style="white-space:pre-wrap">'
                     f'{escape(block["text"])}</pre></details>'))

P066 Private Equity Investment in Behavioral Health Treatment Centers
Status: missing | Method: reviewed_no_abstract
Document SHA-256: 877776674941787619ffa810f3debd18d174bcd7a7e8d2f182061a8fcf5787dd
Candidate: 
Selected spans: []
Abstract source: pdf_abstract 
Cached source: outputs/corpus/extraction/documents/3807949ab7e957570764528178161d9828433b60610c0110bafb743495575b7e.json


## 3. Optional explicit review decision

Leave `REVIEW_ACTION = None` to preserve existing decisions. After inspection, use `select_spans` with exact source ranges, or `no_abstract` with a reason. A decision is bound to the document checksum and stops being valid if extraction changes. This changes only this workflow's review file. It does not alter PDF scope decisions.

For an entire block, use `{'block_id': '...', 'page': 1, 'start': 0, 'end': len(block_text)}`. Multiple spans retain their listed order. Do not select the introduction merely to fill a missing abstract. Rerun the following preparation and embedding cells after an actual review change.

The four approved full-abstract supplements take precedence over earlier PDF-only review decisions. Changing their inclusion requires an explicit change to `configs/selected_abstracts_supplements.json`; PDF review decisions alone do not replace those source records.

In [4]:
REVIEW_ACTION = None  # None, 'select_spans', or 'no_abstract'.
REVIEW_PAPER_ID = INSPECT_PAPER_ID
REVIEW_SPANS = []
REVIEW_REASON = ''
if REVIEW_ACTION is not None:
    reviewed_row = audit.set_index('paper_id').loc[REVIEW_PAPER_ID]
    if REVIEW_ACTION not in {'select_spans', 'no_abstract'} or not REVIEW_REASON.strip():
        raise ValueError('Choose a supported action and record a substantive review reason.')
    proposed = dict(reviewed)
    proposed[reviewed_row.pdf_sha256] = {
        'document_sha256': reviewed_row.document_sha256, 'action': REVIEW_ACTION,
        'spans': REVIEW_SPANS, 'reason': REVIEW_REASON,
    }
    # Validate against actual source spans before saving the decision.
    prepare_selected_abstracts(ROOT, overrides=proposed)
    write_json(OVERRIDES_FILE, proposed)
    reviewed = proposed
    print('Saved reviewed source decision for', REVIEW_PAPER_ID)
else:
    print('No review decisions changed.')

No review decisions changed.


## 4. Save inputs and the full missing-abstract report

Each preparation has a content-derived folder under `outputs/selected_abstracts/prepared/`. It contains the exact ready inputs, an audit of every selected paper, pending records, separate `missing_abstracts.csv` and `extraction_review_needed.csv` reports with IDs/titles/reasons, and input checksums. Earlier preparations are retained.

`supplementary_abstracts.csv` lists all four full-abstract additions with source URLs, content types, checksums and review reasons.

In [5]:
ready, audit, preparation = prepare_selected_abstracts(ROOT, overrides=reviewed)
PREPARED_DIR, preparation_manifest = save_preparation(ROOT, ready, audit, preparation)
print('Prepared inputs and coverage report:', PREPARED_DIR)
print(f"Will embed {len(ready)} of {len(audit)} selected works.")
print(f"Pending for review: {preparation['review_abstracts']}; missing/unavailable: {preparation['missing_abstracts']}")
display(audit.loc[audit.status.ne('ready'), ['paper_id', 'title', 'status', 'review_reason']])

Prepared inputs and coverage report: /Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/outputs/selected_abstracts/prepared/14fe916595b8103d8cd9
Will embed 282 of 292 selected works.
Pending for review: 0; missing/unavailable: 10


,paper_id,title,status,review_reason
64,P066,Private Equity Investment in Behavioral Health...,missing,Viewpoint: title is followed by the article na...
68,P070,Expansion of Private Equity Involvement in Wom...,missing,Research letter: the PDF begins with the end o...
150,P159,"Eileen Appelbaum and Rosemary Batt, Private Eq...",missing,Review Symposium/book review: keywords and rev...
158,P168,Physician Practice Management and Private Equi...,missing,Practice Management article: introductory narr...
159,P169,Gastroenterology Physician Practice Management...,missing,Practice Management follow-up article: introdu...
167,P180,Potential Implications of Private Equity Inves...,missing,Viewpoint: introductory narrative proceeds int...
189,P204,Erratum The financial performance of reverse l...,missing,"The cached PDF is an erratum, containing a cor..."
272,P297,"Commentary on ""The Impact of Vertical Integrat...",missing,"Commentary: title, author details and Keywords..."
274,P299,Geographic Variation in Private Equity Penetra...,missing,Research Letter: title and authors are followe...
291,P318,Private Equity Acquisitions of Physician Medic...,missing,Research Letter: title is followed by backgrou...


## 5. Match the large-corpus embedding method

Settings come from the completed large-corpus run, including its exact model revision. Only the cached revision is loaded. The complete title and abstract are tokenized together, including special tokens, and checked against 8,192 tokens. Overlong inputs stop execution. No vectors are padded, truncated or derived from the old Qwen embeddings.

A completed run for these exact prepared inputs is validated and reused on later executions. Changed source text or review decisions produce a new preparation and run. Source files and prior outputs are retained.

In [6]:
reference, reference_dir = reference_settings(ROOT)
display(pd.DataFrame([{
    'Reference corpus': reference['corpus_size'], 'Model': reference['model'],
    'Revision': reference['model_revision'], 'Dimensions': reference['shape'][1],
    'Pooling': reference['pooling'], 'Normalization': reference['normalization'],
    'Input': reference['input_format'], 'Token limit': reference['max_tokens'],
    'Truncation': reference['truncation'], 'Chunking': reference['chunking'],
}]))

,Reference corpus,Model,Revision,Dimensions,Pooling,Normalization,Input,Token limit,Truncation,Chunking
0,3819,BAAI/bge-m3,5617a9f61b028005a4858fdac845db406aefb181,1024,model-defined CLS,L2,title + blank line + abstract,8192,False,False


## 6. Embed verified abstracts and hand off to clustering

`ALLOW_PARTIAL_CORPUS = True` implements the user's choice to retain missing/uncertain papers in the audit while embedding the approved full abstracts. Set it to `False` to require every selected work before inference. The actual included count is recorded in every run. The missing/uncertain records are not represented by title-only vectors.

`RUN_EMBEDDING = True` runs inference only when an identical completed run is not already available. The local handoff config points explicitly to the authenticated new run and does not change the large-abstract or full-PDF workflows.

In [7]:
RUN_EMBEDDING = True
ALLOW_PARTIAL_CORPUS = True
DEVICE = 'auto'
BATCH_SIZE = 8
selected_run = None
if RUN_EMBEDDING:
    selected_run = embed_selected_abstracts(
        ROOT, PREPARED_DIR, allow_partial=ALLOW_PARTIAL_CORPUS, device=DEVICE, batch_size=BATCH_SIZE,
    )
    config = activate_selected_run(ROOT, selected_run, PREPARED_DIR)
    embedded_papers, vectors, provenance = load_selected_embeddings(selected_run, PREPARED_DIR / 'abstracts_clean.csv')
    print('Saved/reused matrix:', vectors.shape, '| Run:', selected_run)
    print('Active selected-paper abstract input:', config)
    print('Next: Clustering Elite Papers Abstracts.ipynb, then Topic analysis Elite Papers Abstracts.ipynb')
else:
    print('Preparation complete. Enable RUN_EMBEDDING to compute the new vectors.')

/Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 47012.18it/s]

/Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/src/batill/selected_abstracts.py:546: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if model.get_sentence_embedding_dimension() != 1024 or model.max_seq_length < 8192:


Embedded 8/282 selected abstracts


...

Embedded 264/282 selected abstracts


Embedded 272/282 selected abstracts


Embedded 280/282 selected abstracts


Embedded 282/282 selected abstracts


Saved/reused matrix: (282, 1024) | Run: /Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/outputs/selected_abstracts/embeddings/e606ddb5c9ddf172c49d/20261006T113841_841075Z_0d7099bb
Active selected-paper abstract input: {'schema_version': 1, 'run_dir': 'outputs/selected_abstracts/embeddings/e606ddb5c9ddf172c49d/20261006T113841_841075Z_0d7099bb', 'source_csv': 'outputs/selected_abstracts/prepared/14fe916595b8103d8cd9/abstracts_clean.csv', 'corpus_size': 282, 'representation_id': 'e606ddb5c9ddf172c49d450f6426c8da3cc72b7a7c8dbd3bd2e5c1f0662c310e', 'run_summary_sha256': '556c8e47ae6524c216d21fd52e67e4fd5fde7d0174eb9a08cc6e83f519d87f36'}
Next: Clustering Elite Papers Abstracts.ipynb, then Topic analysis Elite Papers Abstracts.ipynb
